In [15]:
# Librerías
import pandas as pd
import numpy as np
import time
import random
from datetime import datetime
from zoneinfo import ZoneInfo

# Funciones del proyecto
import scrapping_functions
from scrapping_functions import get_json_from_url, save_json, parse_json_to_model

# Modelos de datos (pydantic)
import pydantic_model
from pydantic_model import categories, products

##### Categorías

In [16]:
json_response = get_json_from_url(
    url = "https://api.app.biggie.com.py/api/classifications/web?take=-1&storeType=", 
    fixed_user_agent = False
)

[⏱️] Esperando 4.80s...
[🌐] Solicitando: https://api.app.biggie.com.py/api/classifications/web?take=-1&storeType=


In [17]:
save_json(
    data=json_response.get("items", []),
    name="biggie_categories",
    subfolder="biggie/categories"
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/biggie/categories/biggie_categories_2025-07-20_01-37-45.json


In [18]:
if json_response:
    categories_model = parse_json_to_model(
        json_data=json_response,
        model_class=categories,
        supermarket="biggie"
    )

In [19]:
df = pd.DataFrame([c.model_dump() for c in categories_model])
df.head()

,id,name,slug,supermarket,ingestion_time
0,6,Alimentos Especiales,alimentos-especiales,biggie,2025-07-20 01:37:45.298692-03:00
1,1,Almacén,almacen,biggie,2025-07-20 01:37:45.298692-03:00
2,246,Asado,asado,biggie,2025-07-20 01:37:45.298692-03:00
3,41,Bebes,bebes,biggie,2025-07-20 01:37:45.298692-03:00
4,3,Bebidas con Alcohol,bebidas-con-alcohol,biggie,2025-07-20 01:37:45.298692-03:00


##### Productos

In [20]:
categories = df["slug"][16:-3].unique() # [15:-3] subset for testing
np.random.shuffle(categories)
print(f"Number of categories -> {categories.size}")
categories

Number of categories -> 1


array(['mascotas'], dtype=object)

In [9]:
NOW = datetime.now(ZoneInfo("America/Asuncion"))
NumberResults = 24
BASE_URL = "https://api.app.biggie.com.py/api/articles"
all_items = []
total_calls = 0

for i, category in enumerate(categories, start=1):
    print(f"[{i}/{len(categories)}] Scrapeando categoría: {category}")
    skip = 0

    while True:
        url = (
            f"{BASE_URL}?take={NumberResults}&skip={skip}&classificationName={category}"
        )

        wait = random.uniform(1.0, 10.0)
        time.sleep(wait)

        try:
            data = get_json_from_url(
                url = url, 
                use_random_wait = True,
                fixed_user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 ",
                silent = True
            )
        except Exception as e:
            print(f"[❌] Fallo en categoría {category}, skip={skip}: {e}")
            break

        items = data.get("items", [])
        total_calls += 1

        if not items:
            break

        for item in items:
            item["category"] = category
            item["ingestion_time"] = NOW

        all_items.extend(items)
        skip += NumberResults

print(f"\n✅ Fin del scraping: {len(all_items)} productos recolectados en {total_calls} requests.")

[1/1] Scrapeando categoría: mascotas

✅ Fin del scraping: 131 productos recolectados en 7 requests.


In [10]:
save_json(
    data=all_items,
    name="biggie_products",
    subfolder="biggie/products"
)

[💾] Guardado en: /workspaces/tesis-ivan-gennaro/scripts/bronze/outputs/biggie/products/biggie_products_2025-07-20_01-32-15.json


In [12]:
if all_items :
    products_model = parse_json_to_model(
        json_data = all_items, 
        model_class = products, 
        supermarket = 'biggie'
    )

In [13]:
df = pd.DataFrame([c.model_dump() for c in products_model])
df.head()

,code,name,price,category,supermarket,ingestion_time
0,8445290594587,Alimento para Perro Dog Chow Razas Medianas y ...,45800,mascotas,biggie,2025-07-20 01:30:03.416970-03:00
1,8445290595447,Alimento para Perro Dog Chow Razas Pequeñas Ad...,45800,mascotas,biggie,2025-07-20 01:30:03.416970-03:00
2,7797453001564,Alimento para Perro Pedigree Adulto Razas pequ...,45500,mascotas,biggie,2025-07-20 01:30:03.416970-03:00
3,7797453973090,Alimento Para Gato Whiskas Castrados Mix Carne...,47250,mascotas,biggie,2025-07-20 01:30:03.416970-03:00
4,7797453001502,Alimento para Perro Pedigree Cachorro de 1.500...,51900,mascotas,biggie,2025-07-20 01:30:03.416970-03:00
